<a href="https://colab.research.google.com/github/Mumthaz-Alfan/ICT-ACTIVITY/blob/main/Kagglecasestudy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://www.kaggle.com/competitions/predicting-the-sex-of-laysan-albatross/data*italicized text*

In [22]:
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

# Read training and test data
train_df = pd.read_csv("/content/trainsetkaggle.csv")
test_df = pd.read_csv("/content/testtestkaggle.csv")

In [23]:
train_df.info()
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 94 entries, 0 to 93
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              94 non-null     object 
 1   sexo            94 non-null     object 
 2   longitudCraneo  94 non-null     float64
 3   longitudPico    94 non-null     float64
 4   longitudNarina  94 non-null     float64
 5   anchoCraneo     94 non-null     float64
 6   altoPico        94 non-null     float64
 7   anchoPico       94 non-null     float64
 8   tarso           94 non-null     float64
 9   longAlaCerrada  94 non-null     float64
dtypes: float64(8), object(2)
memory usage: 7.5+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              41 non-null     object 
 1   longitudCraneo  41 non-null     float64
 2   longitudPico    4

In [24]:
train_df.head()

,id,sexo,longitudCraneo,longitudPico,longitudNarina,anchoCraneo,altoPico,anchoPico,tarso,longAlaCerrada
0,0J4_2016,H,174.18,106.85,80.19,52.43,30.20,23.26,88.74,51.0
1,A68_2016,H,171.12,104.59,79.45,51.14,30.61,24.04,90.22,50.5
2,E55_2015,H,174.69,106.83,82.14,49.09,30.60,26.47,91.42,52.7
3,7C7_2016,M,182.25,113.17,84.13,52.26,32.08,26.07,91.74,52.6
4,2C0_2018,H,173.70,105.19,80.37,49.20,34.00,27.12,87.77,50.8


In [25]:
display(train_df)

,id,sexo,longitudCraneo,longitudPico,longitudNarina,anchoCraneo,altoPico,anchoPico,tarso,longAlaCerrada
0,0J4_2016,H,174.18,106.85,80.19,52.43,30.20,23.26,88.74,51.0
1,A68_2016,H,171.12,104.59,79.45,51.14,30.61,24.04,90.22,50.5
2,E55_2015,H,174.69,106.83,82.14,49.09,30.60,26.47,91.42,52.7
3,7C7_2016,M,182.25,113.17,84.13,52.26,32.08,26.07,91.74,52.6
4,2C0_2018,H,173.70,105.19,80.37,49.20,34.00,27.12,87.77,50.8
...,...,...,...,...,...,...,...,...,...,...
89,H50_2017,H,168.18,106.70,80.30,49.20,31.30,24.50,91.30,53.4
90,3C0_2015,H,173.95,106.88,81.77,50.27,32.54,25.65,88.64,52.4
91,A62_2016,M,181.01,109.98,82.95,55.47,32.84,25.84,90.78,51.4
92,H24_2015,M,181.51,112.57,85.58,54.11,32.60,26.02,93.11,53.0


In [26]:
train_df.isnull().sum()


,0
id,0
sexo,0
longitudCraneo,0
longitudPico,0
longitudNarina,0
anchoCraneo,0
altoPico,0
anchoPico,0
tarso,0
longAlaCerrada,0


In [27]:
from sklearn.linear_model import LinearRegression

# Fit model on test_df to predict open wing length from closed wing length
wing_model = LinearRegression()
wing_model.fit(test_df[["longAlaCerrada"]], test_df["longAlaAbierta"])

def build_features(df):
    df = df.copy()

    # Impute longAlaAbierta if missing (for train_df)
    if "longAlaAbierta" not in df.columns:
        df["longAlaAbierta"] = wing_model.predict(df[["longAlaCerrada"]])

    # Wing ratio feature
    df["wing_ratio"] = df["longAlaAbierta"] / (df["longAlaCerrada"] + 1e-5)

    # Shape ratios
    df["beak_to_skull_ratio"] = df["longitudPico"] / (df["longitudCraneo"] + 1e-5)
    df["beak_aspect_ratio"] = df["altoPico"] / (df["anchoPico"] + 1e-5)

    # Approximate body volume
    df["body_volume_approx"] = (
        (df["longitudCraneo"] / 10.0) * (df["anchoCraneo"] / 10.0) * df["tarso"]
    )

    return df

# Transform features
train_features = build_features(train_df)
test_features = build_features(test_df)

# Target mappings
target_map = {"H": 0, "M": 1}
rev_target_map = {0: "H", 1: "M"}

In [28]:
drop_cols = ["id", "sexo"]

# Extract features and targets
X = train_features.drop(
    columns=[c for c in drop_cols if c in train_features.columns]
)
y = train_features["sexo"].map(target_map)

# Match X_test columns exactly to X
X_test = test_features[X.columns]

In [29]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
test_probs = np.zeros(len(X_test))
oof_preds = np.zeros(len(X))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    model = LGBMClassifier(
        n_estimators=100, learning_rate=0.05, max_depth=4, random_state=42
    )
    model.fit(X_train, y_train)

    # Collect out-of-fold and test predictions
    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
    test_probs += model.predict_proba(X_test)[:, 1] / skf.n_splits

[LightGBM] [Info] Number of positive: 30, number of negative: 45
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000049 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 318
[LightGBM] [Info] Number of data points in the train set: 75, number of used features: 13
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.400000 -> initscore=-0.405465
[LightGBM] [Info] Start training from score -0.405465
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best ga

In [30]:
best_threshold = 0.5
best_f1 = 0.0

for threshold in np.arange(0.3, 0.7, 0.01):
    binary_preds = (oof_preds >= threshold).astype(int)
    score = f1_score(y, binary_preds, average="macro")
    if score > best_f1:
        best_f1 = score
        best_threshold = threshold

print(f"Optimal Threshold: {best_threshold:.2f}")
print(f"Out-of-Fold Macro F1 Score: {best_f1:.4f}")

Optimal Threshold: 0.47
Out-of-Fold Macro F1 Score: 0.8454


In [31]:
# Convert probabilities to target strings
final_binary_preds = (test_probs >= best_threshold).astype(int)
final_labels = [rev_target_map[val] for val in final_binary_preds]

# Save submission file
submission = pd.DataFrame({"id": test_df["id"], "sexo": final_labels})
submission.to_csv("submission.csv", index=False)

print("Submission successfully saved to submission.csv")

Submission successfully saved to submission.csv
